## Importaciones

In [1]:
import numpy as np
import pandas as pd

## Configuración y Utilidades

In [2]:
# Función signo (igual a Excel: SIGNO)
def sgn(u: float) -> float:
    if u > 0:  return 1.0
    if u < 0:  return -1.0
    return 0.0  # si u==0 devuelve 0

## Datos y Pesos iniciales (manuales) con Early Stop

In [5]:
# ====== Datos (puedes editarlos a mano) ======
X = np.array([
    [-1, -1,  1],
    [-1,  1,  1],
    [ 1, -1,  1],
    [ 1,  1,  1],
], dtype=float)                # X3=1 es el bias explícito (recomendado dejarlo en 1)

Z = np.array([-1, -1, -1, +1], dtype=float)  # AND en bipolar

# ====== Pesos iniciales MANUALES ======
W0 = np.array([2.0, -1.0, 0.5], dtype=float)  # [W1, W2, W3(bias)] cámbialos si quieres

# ====== Hiperparámetros ======
alpha       = 1.0        # en tu Excel α=1
num_epochs  = 5          # pon 5; si converge antes, se detiene con early_stop
use_label_rule = True    # True = regla por etiqueta (Excel). False = regla con error
early_stop  = True       # ✅ detiene cuando termina una época con 100%

## Entrenamiento

In [6]:
def train_excel_like(X, Z, W_init, alpha=1.0, num_epochs=5, use_label_rule=True, early_stop=True):
    W = np.asarray(W_init, float).copy()
    X = np.asarray(X, float); Z = np.asarray(Z, float)

    rows = []
    epochs_summary = []
    converged_epoch = None

    for epoch in range(1, int(num_epochs) + 1):
        correct = 0
        for i, (xi, z) in enumerate(zip(X, Z)):
            W_before = W.copy()

            Sk = float(np.dot(xi, W_before))   # S(k)
            Yk = sgn(Sk)                        # Y(k)
            err = float(z - Yk)                 # Z - Y

            if use_label_rule:
                dW = alpha * z * xi if (Yk != z) else np.zeros_like(xi)
            else:
                dW = alpha * err * xi

            W = W_before + dW
            correct += int(Yk == z)

            rows.append({
                "Época": epoch, "Patrón": i,
                "X1": xi[0], "X2": xi[1], "X3(bias)": xi[2],
                "Z(k)": z, "S(k)": Sk, "Y(k)": Yk, "Error Z-Y": err,
                "W1 antes": W_before[0], "W2 antes": W_before[1], "W3 antes": W_before[2],
                "ΔW1": dW[0], "ΔW2": dW[1], "ΔW3": dW[2],
                "W1 nuevo": W[0], "W2 nuevo": W[1], "W3 nuevo": W[2],
            })

        epochs_summary.append({
            "Época": epoch, "Aciertos (de 4)": correct,
            "Desempeño": f"{100*correct/len(X):.0f}%"
        })

        if early_stop and correct == len(X):
            converged_epoch = epoch
            break

    df_steps = pd.DataFrame(rows)[[
        "Época","Patrón","X1","X2","X3(bias)","Z(k)","S(k)","Y(k)","Error Z-Y",
        "W1 antes","W2 antes","W3 antes","ΔW1","ΔW2","ΔW3",
        "W1 nuevo","W2 nuevo","W3 nuevo"
    ]]
    df_epochs = pd.DataFrame(epochs_summary)
    return df_steps, df_epochs, W, converged_epoch

# ----- EJECUTAR ------
df_steps, df_epochs, W_final, conv_ep = train_excel_like(
    X, Z, W_init=W0, alpha=alpha, num_epochs=num_epochs,
    use_label_rule=use_label_rule, early_stop=early_stop
)

print("== TABLA — Paso a paso (como Excel) ==")
display(df_steps)

print("== TABLA — Resumen por época ==")
display(df_epochs)

if conv_ep is not None:
    print(f"✅ Convergió al 100% en la época {conv_ep}.")
else:
    print("ℹ️ No se alcanzó 100% dentro del número de épocas elegido.")

print("Pesos finales:", np.round(W_final, 4))


== TABLA — Paso a paso (como Excel) ==


,Época,Patrón,X1,X2,X3(bias),Z(k),S(k),Y(k),Error Z-Y,W1 antes,W2 antes,W3 antes,ΔW1,ΔW2,ΔW3,W1 nuevo,W2 nuevo,W3 nuevo
0,1,0,-1.0,-1.0,1.0,-1.0,-0.5,-1.0,0.0,2.0,-1.0,0.5,0.0,0.0,0.0,2.0,-1.0,0.5
1,1,1,-1.0,1.0,1.0,-1.0,-2.5,-1.0,0.0,2.0,-1.0,0.5,0.0,0.0,0.0,2.0,-1.0,0.5
2,1,2,1.0,-1.0,1.0,-1.0,3.5,1.0,-2.0,2.0,-1.0,0.5,-1.0,1.0,-1.0,1.0,0.0,-0.5
3,1,3,1.0,1.0,1.0,1.0,0.5,1.0,0.0,1.0,0.0,-0.5,0.0,0.0,0.0,1.0,0.0,-0.5
4,2,0,-1.0,-1.0,1.0,-1.0,-1.5,-1.0,0.0,1.0,0.0,-0.5,0.0,0.0,0.0,1.0,0.0,-0.5
5,2,1,-1.0,1.0,1.0,-1.0,-1.5,-1.0,0.0,1.0,0.0,-0.5,0.0,0.0,0.0,1.0,0.0,-0.5
6,2,2,1.0,-1.0,1.0,-1.0,0.5,1.0,-2.0,1.0,0.0,-0.5,-1.0,1.0,-1.0,0.0,1.0,-1.5
7,2,3,1.0,1.0,1.0,1.0,-0.5,-1.0,2.0,0.0,1.0,-1.5,1.0,1.0,1.0,1.0,2.0,-0.5
8,3,0,-1.0,-1.0,1.0,-1.0,-3.5,-1.0,0.0,1.0,2.0,-0.5,0.0,0.0,0.0,1.0,2.0,-0.5
9,3,1,-1.0,1.0,1.0,-1.0,0.5,1.0,-2.0,1.0,2.0,-0.5,1.0,-1.0,-1.0,2.0,1.0,-1.5


== TABLA — Resumen por época ==


,Época,Aciertos (de 4),Desempeño
0,1,3,75%
1,2,2,50%
2,3,3,75%
3,4,4,100%


✅ Convergió al 100% en la época 4.
Pesos finales: [ 2.   1.  -1.5]


## Testeo Adicional (Imprime la última Época fina)

In [8]:
# === TESTEO CON APRENDIZAJE (igual a Excel) ===
df_test, df_test_summary, W_after_test, _ = train_excel_like(
    X, Z,
    W_init=W_final,        # arrancamos desde los pesos finales del entrenamiento
    alpha=alpha,
    num_epochs=1,          # una pasada
    use_label_rule=use_label_rule,
    early_stop=False       # en test sólo queremos 1 pasada, sin cortar antes
)

print("== TABLA — Testeo (pasada adicional, con aprendizaje) ==")
display(df_test)

print("== Resumen Testeo ==")
display(df_test_summary)

print("Pesos tras testeo:", np.round(W_after_test, 4))


== TABLA — Testeo (pasada adicional, con aprendizaje) ==


,Época,Patrón,X1,X2,X3(bias),Z(k),S(k),Y(k),Error Z-Y,W1 antes,W2 antes,W3 antes,ΔW1,ΔW2,ΔW3,W1 nuevo,W2 nuevo,W3 nuevo
0,1,0,-1.0,-1.0,1.0,-1.0,-4.5,-1.0,0.0,2.0,1.0,-1.5,0.0,0.0,0.0,2.0,1.0,-1.5
1,1,1,-1.0,1.0,1.0,-1.0,-2.5,-1.0,0.0,2.0,1.0,-1.5,0.0,0.0,0.0,2.0,1.0,-1.5
2,1,2,1.0,-1.0,1.0,-1.0,-0.5,-1.0,0.0,2.0,1.0,-1.5,0.0,0.0,0.0,2.0,1.0,-1.5
3,1,3,1.0,1.0,1.0,1.0,1.5,1.0,0.0,2.0,1.0,-1.5,0.0,0.0,0.0,2.0,1.0,-1.5


== Resumen Testeo ==


,Época,Aciertos (de 4),Desempeño
0,1,4,100%


Pesos tras testeo: [ 2.   1.  -1.5]
